# NB06 — OLS Assumptions: LINE

> **When are OLS estimates valid? What breaks when assumptions fail?**

The four classical OLS assumptions:

| Letter | Assumption | What it means |
|--------|-----------|---------------|
| L | Linearity | True relationship is linear in parameters |
| I | Independence | Observations are independent of each other |
| N | Normality | Residuals are normally distributed |
| E | Equal variance | Residuals have constant variance (homoscedasticity) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(42)
n = 100
X = np.linspace(1, 10, n)

# --- Good data: all assumptions met ---
y_good = 2*X + 5 + np.random.normal(0, 2, n)

# --- L violated: non-linear relationship ---
y_nonlinear = 2*X**2 + np.random.normal(0, 5, n)

# --- E violated: heteroscedasticity ---
y_hetero = 2*X + 5 + np.random.normal(0, X*0.8, n)  # variance grows with X

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, (y_data, title) in enumerate([
    (y_good,       "Good data (all assumptions met)"),
    (y_nonlinear,  "L violated: non-linear data"),
    (y_hetero,     "E violated: heteroscedasticity"),
]):
    m = LinearRegression().fit(X.reshape(-1,1), y_data)
    yhat = m.predict(X.reshape(-1,1))
    resid = y_data - yhat

    ax_top = axes[0, col]
    ax_top.scatter(X, y_data, s=15, alpha=0.5, color='steelblue')
    ax_top.plot(X, yhat, 'r-', linewidth=2)
    ax_top.set_title(title); ax_top.grid(alpha=0.3)

    ax_bot = axes[1, col]
    ax_bot.scatter(yhat, resid, s=15, alpha=0.5, color='orange')
    ax_bot.axhline(0, color='red', linewidth=1)
    ax_bot.set_xlabel('Fitted values'); ax_bot.set_ylabel('Residuals')
    ax_bot.set_title('Residuals vs Fitted'); ax_bot.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Top row: scatter + fit.  Bottom row: residuals vs fitted (key diagnostic plot).")
print("Good data: random scatter around 0.")
print("Non-linear: curved pattern in residuals.")
print("Hetero: funnel pattern — variance increases with fitted value.")


## L — Linearity

**Assumption:** E[y | x] = β₀ + β₁x  (the mean of y is a linear function of x)

**What breaks:** biased coefficients, biased predictions

**How to detect:** residuals vs fitted — look for a curved pattern

**Fix:** add polynomial terms, log-transform, or use a non-linear model


## I — Independence

**Assumption:** ε₁, ε₂, ..., εₙ are independent of each other

**What breaks:** standard errors are wrong → t-tests and CIs are invalid

**Common causes:** time-series data, clustered data, repeated measures

**How to detect:** Durbin-Watson test, ACF plot of residuals

**Fix:** use time-series models (ARIMA), mixed effects models, clustered SEs


In [ ]:
# Demonstrate autocorrelation in residuals
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.stattools import durbin_watson

np.random.seed(0)
n = 100
X = np.linspace(0, 10, n)

# Independent residuals (good)
e_indep = np.random.normal(0, 1, n)

# Autocorrelated residuals (bad) — AR(1) process
e_auto = np.zeros(n)
e_auto[0] = np.random.normal()
for i in range(1, n):
    e_auto[i] = 0.85 * e_auto[i-1] + np.random.normal(0, 0.5)

y_indep = 2*X + 5 + e_indep
y_auto  = 2*X + 5 + e_auto

from sklearn.linear_model import LinearRegression
resid_indep = y_indep - LinearRegression().fit(X.reshape(-1,1), y_indep).predict(X.reshape(-1,1))
resid_auto  = y_auto  - LinearRegression().fit(X.reshape(-1,1), y_auto).predict(X.reshape(-1,1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, resid, title in zip(axes,
    [resid_indep, resid_auto],
    ['Independent residuals', 'Autocorrelated residuals (AR1 rho=0.85)']):
    ax.plot(resid, color='steelblue', linewidth=0.8)
    ax.axhline(0, color='red', linewidth=1)
    ax.set_title(title); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

dw_i = durbin_watson(resid_indep)
dw_a = durbin_watson(resid_auto)
print(f"Durbin-Watson (independent):    {dw_i:.3f}  (want ~2.0)")
print(f"Durbin-Watson (autocorrelated): {dw_a:.3f}  (<2 = positive autocorrelation)")


## N — Normality of residuals

**Assumption:** residuals ~ Normal(0, σ²)

**What breaks:** t-tests and p-values are approximate (not exact).
For large n, CLT saves you. For small n, this matters more.

**How to detect:** Q-Q plot of residuals; Shapiro-Wilk test

**Fix:** transform y (log, Box-Cox); robust regression


In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(1)
n = 80

# Normal residuals
e_normal = np.random.normal(0, 1, n)
# Skewed residuals
e_skewed = np.random.exponential(1, n) - 1

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, resid, title in zip(axes,
    [e_normal, e_skewed],
    ['Normal residuals', 'Skewed residuals']):
    (osm, osr), (slope, intercept, r) = stats.probplot(resid)
    ax.scatter(osm, osr, s=20, color='steelblue', alpha=0.7)
    xs = np.array([osm.min(), osm.max()])
    ax.plot(xs, slope*xs+intercept, 'r-', linewidth=2, label='Normal line')
    sw_stat, sw_p = stats.shapiro(resid)
    ax.set_title(f'{title}\nShapiro-Wilk p={sw_p:.4f}')
    ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Sample quantiles')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Q-Q Plots: points on the diagonal = normal residuals')
plt.tight_layout(); plt.show()


## E — Equal variance (Homoscedasticity)

**Assumption:** Var(εᵢ) = σ² for all i (constant variance)

**What breaks:** SEs are wrong → invalid t-tests and CIs.
Predictions are still unbiased, but inefficient.

**How to detect:** scale-location plot; Breusch-Pagan test

**Fix:** weighted least squares (WLS); log-transform y; robust SEs (HC3)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

np.random.seed(3)
n = 100
X = np.linspace(1, 10, n)

# Homoscedastic
y_homo  = 2*X + 5 + np.random.normal(0, 2, n)
# Heteroscedastic
y_hetero = 2*X + 5 + np.random.normal(0, 0.5*X, n)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_data, title in zip(axes,
    [y_homo, y_hetero],
    ['Homoscedastic', 'Heteroscedastic']):
    Xsm = sm.add_constant(X)
    res = sm.OLS(y_data, Xsm).fit()
    std_resid = np.sqrt(np.abs(res.resid / res.resid.std()))
    ax.scatter(res.fittedvalues, std_resid, s=20, alpha=0.6, color='steelblue')
    ax.set_xlabel('Fitted values'); ax.set_ylabel('√|Standardised residuals|')
    bp_stat, bp_p, *_ = het_breuschpagan(res.resid, Xsm)
    ax.set_title(f'{title}\nBreusch-Pagan p={bp_p:.4f}')
    ax.grid(alpha=0.3)

plt.suptitle('Scale-Location Plot: flat band = homoscedastic')
plt.tight_layout(); plt.show()
print("Breusch-Pagan p < 0.05 → reject homoscedasticity")


## Summary

| Assumption | Key test | Fix |
|-----------|---------|-----|
| Linearity | Residuals vs Fitted (curved?) | Polynomial, log transform |
| Independence | Durbin-Watson (~2), ACF plot | Time-series model, cluster SEs |
| Normality | Q-Q plot, Shapiro-Wilk | Log/Box-Cox transform |
| Equal variance | Scale-Location, Breusch-Pagan | WLS, robust SEs |

**Next:** NB07 — diagnostic plots in detail (4 standard plots from R's `plot(lm)`)
